In [ ]:
import pandas as pd
import numpy
import json
import os
from dotenv import load_dotenv
load_dotenv()
os.chdir(os.getenv('PARENT_DIR'))

In [ ]:
from copy import deepcopy

In [ ]:
orders = ['aos', 'aso', 'sao', 'oas', 'osa']

In [ ]:
dataset_type = 'hoasa_hotel_aug'
lang = 'indo'
ori_data_path = f'dataset/{dataset_type}/{lang}/mvp_aos/train.json'
with open(ori_data_path, 'r') as f:
    ori_data = json.load(f)

In [ ]:
from typing import List, Dict
import re
def parse_absa_string(text: str) -> List[Dict[str, str]]:
	"""
	Parses a string formatted as "[A] aspect [O] opinion [S] sentiment" into a list of dictionaries.
	Each dictionary contains the tag as the key and the corresponding value.
	For example, "[A] [O] [S] [A] harga [O] terjangkau [S] positive [SSEP] [A] fasilitas [O] nyaman [S] positive" becomes:
	[{'A': 'harga', 'S': 'positive', 'O': 'terjangkau'},
	{'A': 'fasilitas', 'S': 'positive', 'O': 'nyaman'}].

	Args:
		text (str): ABSA string output to be parsed.

	Returns:
		List[Dict[str, str]]: List of dictionaries of parsed ABSA output.

	"""
	pattern = r"\[(\w+)\]\s*([^[]+)"
	matches = re.findall(pattern, text)

	result = []
	current_dict = {}

	for tag, content in matches:
		if tag == "SSEP":  # Sentence separator -> Start a new dictionary
			result.append(current_dict)
			current_dict = {}
		else:
			current_dict[tag] = content.strip()

	if current_dict:  # Append the last sentence if it exists
		result.append(current_dict)

	return result

def convert_output_to_mvp_format(data_list: List[Dict[str, str]], order='aos') -> str:
	"""
	Converts a list of aspect-based sentiment dictionaries to the 
	extraction-style string format used in the MVP paper.

	Args:
		data_list (List[Dict[str, str]]): A list of strings, where each dictionary must contain 
				the keys 'A' (Aspect), 'O' (Opinion), and 'S' (Sentiment).
		order (str): A string specifying the order of elements (e.g., 'aos', 'ao', 'as', 'a', 'o')

	Returns:
		A single string formatted with special tokens and elements based on the order
	"""

	result_str = []
	for triplet in data_list:
		triplet_str = [f"[{element.upper()}] {triplet[element.upper()]}" for element in order]
		triplet_str = ' '.join(triplet_str)
		result_str.append(triplet_str)
	return ' [SSEP] '.join(result_str)

def convert_input_to_mvp_format(input_str: str, order='aos') -> str:
	order_str = ' '.join([f"[{element.upper()}]" for element in order])
	final_str = input_str.replace('[A] [O] [S]', order_str)
	return final_str

In [ ]:
print(convert_output_to_mvp_format(parse_absa_string('[A] harga [O] terjangkau [S] positive [SSEP] [A] fasilitas [O] nyaman [S] positive'), order='aso'))
print(convert_input_to_mvp_format('test 1 2 3 [A] [O] [S]', order='ao'))

In [ ]:
ori_data[0]

In [ ]:
len(ori_data)

In [ ]:
new_data = []
instance_id = 0
for instance in ori_data:
	for order in orders:
		new_data.append({
			'sentence_id': instance['sentence_id'],
			'instance_id': instance_id,
			'input': convert_input_to_mvp_format(instance['input'], order),
			'target': convert_output_to_mvp_format(parse_absa_string(instance['target']), order),
			'element_order': order,
			'task_elements': 'aos',
			'dataset_type': instance['dataset_type']
		})
		instance_id += 1
assert len(new_data) == len(ori_data) * len(orders)
print(len(new_data))

In [ ]:
test = [1,2,3]
test[5:9]

In [ ]:
new_path = f'dataset/{dataset_type}/{lang}/mvp/train.json'
os.makedirs(os.path.dirname(new_path), exist_ok=True)
with open(new_path, 'w') as f:
    json.dump(new_data, f, indent=4, ensure_ascii=False)
print(f"Saved augmented data to {new_path}")